In [5]:
from torch.utils.data import Dataset
import torch
from PIL import Image
import glob
import cv2

In [6]:


class ImageDataset(Dataset):
    def __init__(self, img_dir_pattern, transform=None):
        self.image_paths = glob.glob(img_dir_pattern)
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        # Load image as PIL Image (not numpy array!)
        image = Image.open(img_path).convert('RGB')
        
        # Apply transforms if provided
        if self.transform:
            image = self.transform(image)
        
        return image, img_path

print("ImageDataset class created successfully!")

ImageDataset class created successfully!


In [18]:
from torchvision import transforms
from torch.utils.data import DataLoader


# Define transforms for preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to consistent size
    transforms.ToTensor()
])

# Create dataset and dataloader
img_dir = '../data/normalized_images/normalized_images/*.jpg'
dataset = ImageDataset(img_dir, transform=transform)
dataloader = DataLoader(dataset, 
                       batch_size=32, 
                       shuffle=False,  # Keep order for clustering
                       num_workers=0)  # Set to 0 for Windows compatibility

print(f"Dataset created with {len(dataset)} images")
print(f"DataLoader created with batch size: {dataloader.batch_size}")

# Test loading a few images
sample_batch = next(iter(dataloader))
images, paths = sample_batch
print(f"Sample batch shape: {images.shape}")
print(f"First few image paths: {paths[:3]}")



Dataset created with 7557 images
DataLoader created with batch size: 32
Sample batch shape: torch.Size([32, 3, 224, 224])
First few image paths: ('../data/normalized_images/normalized_images\\img_1.jpg', '../data/normalized_images/normalized_images\\img_10.jpg', '../data/normalized_images/normalized_images\\img_100.jpg')


In [8]:
import torch.nn as nn

class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        # Latent space
        self.fc_mu = nn.Linear(latent_dim, latent_dim)
        self.fc_var = nn.Linear(latent_dim, latent_dim)
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        log_var = self.fc_var(h)
        return mu, log_var

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

In [15]:
import torch.nn.functional as F

def vae_loss(reconstructed_x, x, mu, log_var):
    reconstruction_loss = F.mse_loss(reconstructed_x, x, reduction='sum')
    kl_divergence = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return reconstruction_loss + kl_divergence

In [16]:
def train_vae(model, dataloader, optimizer, num_epochs=20):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for data in dataloader:
            inputs, _ = data
            optimizer.zero_grad()
            reconstructed_x, mu, log_var = model(inputs)
            loss = vae_loss(reconstructed_x, inputs, mu, log_var)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader)}")

In [17]:
model = VAE(input_dim=224*224*3, hidden_dim=512, latent_dim=128)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_vae(model, dataloader, optimizer, num_epochs=20)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (21504x224 and 150528x512)

In [12]:
from sklearn.cluster import KMeans

def cluster_embeddings(embeddings, n_clusters=10):
    kmeans = KMeans(n_clusters=n_clusters)
    clusters = kmeans.fit_predict(embeddings)
    return clusters

In [14]:
%pip install matplotlib

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def visualize_clusters(embeddings, clusters):
    tsne = TSNE(n_components=2)
    reduced_embeddings = tsne.fit_transform(embeddings)
    plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], c=clusters)
    plt.show()

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/7.8 MB ? eta -:--:--
   ---------------------- ----------------- 4.5/7.8 MB 24.4 MB/s eta 0:00:01
   ---------------------------------------- 7.8/7.8 MB 25.5 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 27.0 MB/s  0:00:00
Using cached importlib_resources-6.5.2-py3-none-any.whl (37 kB)
Using cached pyparsing-3.2.3-py3-none-any.whl (111 kB)

   ----- ---------------------------------- 1/7 [kiwisolver]
   ----------- ---------------------------- 2/7 [importlib-resources]
   ----------------- ---------------------- 3/7 [fonttools]
   ----------------- ---------------------- 3/7 [fonttools]
   ----